In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Customer Segmentation using K-means Clustering\n",
    "\n",
    "## Project Overview\n",
    "This notebook demonstrates customer segmentation using K-means clustering algorithm on mall customer data.\n",
    "\n",
    "## Business Objective\n",
    "Segment customers into distinct groups based on purchasing behavior to enable targeted marketing strategies."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✅ Data loaded successfully!\n",
      "📊 Dataset shape: (200, 5)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>CustomerID</th>\n",
       "      <th>Gender</th>\n",
       "      <th>Age</th>\n",
       "      <th>Annual Income (k$)</th>\n",
       "      <th>Spending Score (1-100)</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>1</td>\n",
       "      <td>Male</td>\n",
       "      <td>19</td>\n",
       "      <td>15</td>\n",
       "      <td>39</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>2</td>\n",
       "      <td>Male</td>\n",
       "      <td>21</td>\n",
       "      <td>15</td>\n",
       "      <td>81</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>3</td>\n",
       "      <td>Female</td>\n",
       "      <td>20</td>\n",
       "      <td>16</td>\n",
       "      <td>6</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>4</td>\n",
       "      <td>Female</td>\n",
       "      <td>23</td>\n",
       "      <td>16</td>\n",
       "      <td>77</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>5</td>\n",
       "      <td>Female</td>\n",
       "      <td>31</td>\n",
       "      <td>17</td>\n",
       "      <td>40</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)\n",
       "0           1    Male   19                  15                      39\n",
       "1           2    Male   21                  15                      81\n",
       "2           3  Female   20                  16                       6\n",
       "3           4  Female   23                  16                      77\n",
       "4           5  Female   31                  17                      40"
      ]
     },
     "execution_count": 1,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# Import required libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.cluster import KMeans\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.decomposition import PCA\n",
    "from sklearn.metrics import silhouette_score\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style for better visualizations\n",
    "plt.style.use('seaborn-v0_8')\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "# Load the data\n",
    "df = pd.read_csv('../datasets/mall_customers.csv')\n",
    "print(\"✅ Data loaded successfully!\")\n",
    "print(f\"📊 Dataset shape: {df.shape}\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "🔍 Exploratory Data Analysis\n",
      "==================================================\n",
      "Dataset Info:\n",
      "<class 'pandas.core.frame.DataFrame'>\n",
      "RangeIndex: 200 entries, 0 to 199\n",
      "Data columns (total 5 columns):\n",
      " #   Column                  Non-Null Count  Dtype \n",
      "---  ------                  --------------  ----- \n",
      " 0   CustomerID              200 non-null    int64 \n",
      " 1   Gender                  200 non-null    object\n",
      " 2   Age                     200 non-null    int64 \n",
      " 3   Annual Income (k$)      200 non-null    int64 \n",
      " 4   Spending Score (1-100)  200 non-null    int64 \n",
      "dtypes: int64(4), object(1)\n",
      "memory usage: 7.9+ KB\n",
      "None\n",
      "\n",
      "Missing Values:\n",
      "CustomerID                0\n",
      "Gender                    0\n",
      "Age                       0\n",
      "Annual Income (k$)        0\n",
      "Spending Score (1-100)    0\n",
      "dtype: int64\n",
      "\n",
      "Statistical Summary:\n",
      "       CustomerID         Age  Annual Income (k$)  Spending Score (1-100)\n",
      "count  200.000000  200.000000          200.000000              200.000000\n",
      "mean   100.500000   38.850000           60.560000               50.200000\n",
      "std     57.879185   13.969007           26.264721               25.823522\n",
      "min      1.000000   18.000000           15.000000                1.000000\n",
      "25%     50.750000   28.750000           41.500000               34.750000\n",
      "50%    100.500000   36.000000           61.500000               50.000000\n",
      "75%    150.250000   49.000000           78.000000               73.000000\n",
      "max    200.000000   70.000000          137.000000               99.000000\n"
     ]
    },
    {
     "data": {
      "image/png": "...